# Multi-Agent Conversation — Two Agents Talking

## What This Notebook Teaches You

A single agent can accomplish a lot — but some tasks benefit from **multiple perspectives**. When a Researcher argues with a Skeptic, the result is more rigorous than either alone. When a Teacher explains to a Student, the dialogue forces clarity.

In this notebook, you will:

1. **Build a MessageBus** — structured communication between agents
2. **Build ConversableAgent** — an agent that maintains its own conversation history
3. **Build TwoAgentChat** — orchestrated turn-taking between two agents
4. **Experiment 1**: Researcher + Skeptic debate a scientific claim
5. **Experiment 2**: Teacher + Student explain a concept through dialogue
6. **Experiment 3**: Planner + Critic refine a plan through feedback
7. **Analyze** conversation dynamics — turns to resolution, quality, cost
8. **Compare** shared vs separate context

---

> **Prerequisites**: Notebooks 01–05 (agents, tools, ReAct).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Multiple Agents?

### The Case for Specialization

Single agents work well for straightforward tasks, but struggle when:

- **Multiple perspectives are needed** — a security reviewer and performance reviewer see different issues
- **Verification is critical** — having one agent check another's work catches errors
- **The task has natural roles** — teacher/student, writer/editor, proposer/critic
- **Debate improves quality** — adversarial dialogue forces deeper reasoning

### Research Evidence

Recent papers show multi-agent systems outperforming single agents:

- **Du et al. (2023)** — multi-agent debate improves mathematical and strategic reasoning
- **Liang et al. (2023)** — agents playing different roles produce more diverse and accurate outputs
- **Chan et al. (2023)** — "ChatEval" shows multi-agent discussion improves open-ended text evaluation

### What We'll Build

```
┌─────────────┐     MessageBus     ┌─────────────┐
│  Agent A     │ ←───────────────→ │  Agent B     │
│  (Researcher)│    Messages with   │  (Skeptic)   │
│  own history │    sender/recipient│  own history  │
│  own persona │    timestamps      │  own persona  │
└─────────────┘                    └─────────────┘
        ↕                                  ↕
    TwoAgentChat: orchestrates turn-taking, 
    termination, and conversation logging
```

## 2. Message Infrastructure

### 2.1 — Message Dataclass

Every inter-agent message is a structured object with sender, recipient, content, and timestamp.

In [ ]:
@dataclass
class Message:
    """A structured message between agents."""
    sender: str
    recipient: str
    content: str
    timestamp: float = field(default_factory=time.time)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __str__(self):
        return f"[{self.sender} → {self.recipient}]: {self.content[:100]}{'...' if len(self.content) > 100 else ''}"


class MessageBus:
    """Central message routing system for multi-agent communication."""

    def __init__(self):
        self.messages: List[Message] = []
        self.subscribers: Dict[str, list] = defaultdict(list)

    def send(self, message: Message):
        """Send a message and notify subscribers."""
        self.messages.append(message)
        # Notify recipient's callbacks
        for callback in self.subscribers.get(message.recipient, []):
            callback(message)

    def subscribe(self, agent_name: str, callback: Callable):
        """Register a callback for when agent_name receives a message."""
        self.subscribers[agent_name].append(callback)

    def get_history(self, agent_name: str = None) -> List[Message]:
        """Get message history, optionally filtered by participant."""
        if agent_name is None:
            return self.messages
        return [m for m in self.messages
                if m.sender == agent_name or m.recipient == agent_name]

    def get_conversation(self, agent_a: str, agent_b: str) -> List[Message]:
        """Get all messages between two specific agents."""
        return [m for m in self.messages
                if (m.sender == agent_a and m.recipient == agent_b) or
                   (m.sender == agent_b and m.recipient == agent_a)]

    def stats(self):
        """Get messaging statistics."""
        senders = Counter(m.sender for m in self.messages)
        avg_len = sum(len(m.content) for m in self.messages) / len(self.messages) if self.messages else 0
        return {
            "total_messages": len(self.messages),
            "unique_senders": len(senders),
            "messages_per_sender": dict(senders),
            "avg_message_length": round(avg_len, 0)
        }

# Test the message infrastructure
bus = MessageBus()

msg1 = Message(sender="Alice", recipient="Bob", content="What do you think about transformer architectures?")
msg2 = Message(sender="Bob", recipient="Alice", content="They revolutionized NLP through self-attention mechanisms.")

bus.send(msg1)
bus.send(msg2)

print("All messages:")
for m in bus.messages:
    print(f"  {m}")
print(f"\nStats: {bus.stats()}")

## 3. ConversableAgent — An Agent That Converses

Each agent maintains its own conversation history and personality through a system prompt. When it receives a message, it generates a response considering the full dialogue so far.

In [ ]:
class ConversableAgent:
    """An agent capable of multi-turn conversation with other agents."""

    def __init__(self, name: str, system_prompt: str, bus: MessageBus,
                 max_tokens: int = 300, temperature: float = 0.7):
        self.name = name
        self.system_prompt = system_prompt
        self.bus = bus
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.history: List[Message] = []  # This agent's conversation history
        self.token_count = 0

        # Subscribe to messages addressed to this agent
        self.bus.subscribe(self.name, self._on_message)

    def _on_message(self, message: Message):
        """Callback when this agent receives a message."""
        self.history.append(message)

    def _build_messages(self, conversation_with: str = None):
        """Build the LLM message list from conversation history."""
        messages = [{"role": "system", "content": self.system_prompt}]

        # Add conversation history
        for msg in self.history:
            if msg.sender == self.name:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                messages.append({"role": "user", "content": msg.content})

        return messages

    def respond(self, to: str) -> Message:
        """Generate a response to the latest message in conversation."""
        messages = self._build_messages(conversation_with=to)

        response_text = generate(
            messages,
            max_new_tokens=self.max_tokens,
            temperature=self.temperature
        )

        # Track approximate token usage
        self.token_count += len(response_text.split()) * 1.3  # rough estimate

        # Create and send response message
        msg = Message(sender=self.name, recipient=to, content=response_text)
        self.history.append(msg)
        self.bus.send(msg)
        return msg

    def initiate(self, to: str, message: str) -> Message:
        """Start a conversation with another agent."""
        msg = Message(sender=self.name, recipient=to, content=message)
        self.history.append(msg)
        self.bus.send(msg)
        return msg

    def reset(self):
        """Clear conversation history."""
        self.history = []
        self.token_count = 0

    def stats(self):
        return {
            "name": self.name,
            "messages_sent": sum(1 for m in self.history if m.sender == self.name),
            "messages_received": sum(1 for m in self.history if m.sender != self.name),
            "approx_tokens": int(self.token_count)
        }

# Quick test
test_bus = MessageBus()
agent_a = ConversableAgent(
    name="Curious",
    system_prompt="You are a curious scientist who asks insightful questions. Keep responses under 3 sentences.",
    bus=test_bus
)
agent_b = ConversableAgent(
    name="Expert",
    system_prompt="You are a knowledgeable expert who gives clear, concise answers. Keep responses under 3 sentences.",
    bus=test_bus
)

# One exchange
agent_a.initiate("Expert", "What makes quantum computing fundamentally different from classical computing?")
response = agent_b.respond("Curious")
print(f"Expert replied: {response.content[:200]}...")
print(f"\nAgent stats: {agent_a.stats()}")
print(f"Agent stats: {agent_b.stats()}")

## 4. TwoAgentChat — Orchestrated Dialogue

The TwoAgentChat controller manages turn-taking between two agents, handles termination conditions, and logs the full conversation.

In [ ]:
class TwoAgentChat:
    """Orchestrates a conversation between two agents."""

    def __init__(self, agent_a: ConversableAgent, agent_b: ConversableAgent,
                 max_turns: int = 10):
        self.agent_a = agent_a
        self.agent_b = agent_b
        self.max_turns = max_turns
        self.turns: List[Dict] = []
        self.terminated_reason = None

    def _check_termination(self, message: Message, turn_num: int) -> Optional[str]:
        """Check if conversation should end."""
        content_lower = message.content.lower()

        # Check for explicit agreement/conclusion signals
        conclusion_phrases = [
            "i agree with your conclusion",
            "we have reached consensus",
            "in summary, we both agree",
            "i think we've covered this thoroughly",
            "that's a comprehensive answer",
        ]
        for phrase in conclusion_phrases:
            if phrase in content_lower:
                return f"Consensus reached at turn {turn_num}"

        # Check for max turns
        if turn_num >= self.max_turns:
            return f"Max turns ({self.max_turns}) reached"

        return None

    def run(self, initial_message: str, verbose: bool = True) -> Dict:
        """Run the conversation."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"TWO-AGENT CHAT: {self.agent_a.name} ↔ {self.agent_b.name}")
            print(f"Max turns: {self.max_turns}")
            print(f"{'='*70}")

        # Reset agents
        self.agent_a.reset()
        self.agent_b.reset()
        self.turns = []

        # Agent A initiates
        msg = self.agent_a.initiate(self.agent_b.name, initial_message)
        self.turns.append({"turn": 0, "speaker": self.agent_a.name, "content": msg.content})
        if verbose:
            print(f"\n[Turn 0] {self.agent_a.name}:")
            print(f"  {msg.content[:300]}{'...' if len(msg.content) > 300 else ''}")

        # Alternating turns
        current_responder = self.agent_b
        current_target = self.agent_a
        turn = 1

        start_time = time.time()

        while turn <= self.max_turns:
            # Current agent responds
            msg = current_responder.respond(current_target.name)
            self.turns.append({
                "turn": turn,
                "speaker": current_responder.name,
                "content": msg.content
            })

            if verbose:
                print(f"\n[Turn {turn}] {current_responder.name}:")
                print(f"  {msg.content[:300]}{'...' if len(msg.content) > 300 else ''}")

            # Check termination
            reason = self._check_termination(msg, turn)
            if reason:
                self.terminated_reason = reason
                if verbose:
                    print(f"\n--- Conversation ended: {reason} ---")
                break

            # Swap roles
            current_responder, current_target = current_target, current_responder
            turn += 1

        elapsed = time.time() - start_time

        if not self.terminated_reason:
            self.terminated_reason = f"Max turns ({self.max_turns}) reached"

        result = {
            "agents": [self.agent_a.name, self.agent_b.name],
            "total_turns": len(self.turns),
            "termination_reason": self.terminated_reason,
            "elapsed_seconds": round(elapsed, 1),
            "total_words": sum(len(t['content'].split()) for t in self.turns),
            "agent_a_stats": self.agent_a.stats(),
            "agent_b_stats": self.agent_b.stats(),
        }

        if verbose:
            print(f"\n{'='*70}")
            print(f"CONVERSATION SUMMARY")
            print(f"  Turns: {result['total_turns']}")
            print(f"  Words: {result['total_words']}")
            print(f"  Time: {result['elapsed_seconds']}s")
            print(f"  Reason: {result['termination_reason']}")
            print(f"{'='*70}")

        return result

print("TwoAgentChat class ready")

## 5. Experiment 1: Researcher + Skeptic

A **Researcher** makes a scientific claim and presents evidence. A **Skeptic** challenges assumptions, asks for evidence, and points out weaknesses. This adversarial dynamic produces more rigorous analysis.

In [ ]:
# Create message bus and agents
bus1 = MessageBus()

researcher = ConversableAgent(
    name="Researcher",
    system_prompt=(
        "You are a scientific researcher presenting findings. You make evidence-based claims, "
        "cite studies and data, and defend your position with logical arguments. When challenged, "
        "you either provide additional evidence or acknowledge limitations honestly. "
        "Keep each response to 3-5 sentences. Stay focused on the scientific evidence."
    ),
    bus=bus1,
    max_tokens=250
)

skeptic = ConversableAgent(
    name="Skeptic",
    system_prompt=(
        "You are a scientific skeptic and critical thinker. Your role is to challenge claims, "
        "ask for evidence, identify logical fallacies, point out confounding variables, and "
        "demand rigorous proof. You are not contrarian — you follow evidence, but require strong "
        "evidence before accepting claims. "
        "Keep each response to 3-5 sentences. Ask specific, probing questions."
    ),
    bus=bus1,
    max_tokens=250
)

chat1 = TwoAgentChat(researcher, skeptic, max_turns=8)

result1 = chat1.run(
    "I believe that large language models exhibit emergent abilities at certain scale "
    "thresholds — capabilities that appear suddenly rather than gradually as models get larger. "
    "Research from Google and others has documented abilities like chain-of-thought reasoning "
    "that seem absent in smaller models but emerge in models above 100 billion parameters."
)

### Experiment 1 Analysis

Notice how the Skeptic forces the Researcher to:
- Provide specific evidence and citations
- Address potential confounders
- Acknowledge uncertainty
- Distinguish correlation from causation

This is exactly why multi-agent debate works — the adversarial pressure improves output quality.

## 6. Experiment 2: Teacher + Student

A **Teacher** explains a concept, and a **Student** asks questions, requests clarification, and tries to summarize understanding. The dialogue naturally produces clear, accessible explanations.

In [ ]:
bus2 = MessageBus()

teacher = ConversableAgent(
    name="Teacher",
    system_prompt=(
        "You are a patient, skilled teacher explaining complex concepts simply. "
        "Use analogies, examples, and build understanding step by step. "
        "When the student asks questions, address them directly. "
        "Check understanding by asking the student to explain back. "
        "Keep each response to 3-5 sentences."
    ),
    bus=bus2,
    max_tokens=250
)

student = ConversableAgent(
    name="Student",
    system_prompt=(
        "You are a curious, engaged student learning a new concept. "
        "Ask clarifying questions when something is unclear. "
        "Try to connect new ideas to things you already know. "
        "Periodically summarize your understanding and ask if you've got it right. "
        "Admit when you're confused. Keep each response to 2-4 sentences."
    ),
    bus=bus2,
    max_tokens=200
)

chat2 = TwoAgentChat(teacher, student, max_turns=8)

result2 = chat2.run(
    "Today I'd like to explain how neural networks learn through backpropagation. "
    "Think of it this way: imagine you're trying to throw a ball into a basket. "
    "Each time you miss, you adjust your throw based on how far off you were. "
    "Backpropagation is essentially that adjustment process for neural networks."
)

## 7. Experiment 3: Planner + Critic

A **Planner** proposes a plan for a complex task. A **Critic** identifies weaknesses, risks, and missing steps. The Planner then revises. This iterative refinement produces better plans.

In [ ]:
bus3 = MessageBus()

planner = ConversableAgent(
    name="Planner",
    system_prompt=(
        "You are a strategic planner who creates detailed, actionable plans. "
        "Start with a high-level overview, then break tasks into specific steps. "
        "When receiving criticism, revise your plan to address the weaknesses. "
        "Be concrete — include timelines, resources, and success metrics. "
        "Keep each response to 4-6 sentences."
    ),
    bus=bus3,
    max_tokens=300
)

critic = ConversableAgent(
    name="Critic",
    system_prompt=(
        "You are a project critic who identifies weaknesses in plans. "
        "Look for: missing steps, unrealistic timelines, resource gaps, risk factors, "
        "dependencies that could block progress, and unclear success criteria. "
        "Be specific and constructive — don't just criticize, suggest improvements. "
        "Keep each response to 3-5 sentences."
    ),
    bus=bus3,
    max_tokens=250
)

chat3 = TwoAgentChat(planner, critic, max_turns=8)

result3 = chat3.run(
    "Here's my plan for building a recommendation system for an e-commerce platform: "
    "Phase 1 (2 weeks): Collect and clean user interaction data including clicks, purchases, "
    "and ratings. Phase 2 (3 weeks): Implement collaborative filtering using matrix factorization. "
    "Phase 3 (2 weeks): Build a REST API and integrate with the frontend. "
    "Phase 4 (1 week): A/B test against the current system and measure conversion rate improvement."
)

## 8. Conversation Analysis

Let's compare the three experiments quantitatively.

In [ ]:
# Compare all three experiments
experiments = [
    ("Researcher vs Skeptic", result1),
    ("Teacher vs Student", result2),
    ("Planner vs Critic", result3),
]

print(f"{'Experiment':<30s} {'Turns':>6s} {'Words':>7s} {'Time':>7s} {'Termination'}")
print("-" * 85)
for name, result in experiments:
    print(f"{name:<30s} {result['total_turns']:>6d} {result['total_words']:>7d} "
          f"{result['elapsed_seconds']:>6.1f}s {result['termination_reason']}")

# Per-agent analysis
print(f"\n{'Agent':<20s} {'Messages Sent':>15s} {'Approx Tokens':>15s}")
print("-" * 55)
for name, result in experiments:
    for agent_key in ['agent_a_stats', 'agent_b_stats']:
        stats = result[agent_key]
        print(f"  {stats['name']:<18s} {stats['messages_sent']:>13d} {stats['approx_tokens']:>13d}")

# Words per turn distribution
print(f"\nWords per turn:")
for name, result in experiments:
    if result['total_turns'] > 0:
        avg_words = result['total_words'] / result['total_turns']
        print(f"  {name}: {avg_words:.0f} words/turn avg")

### 8.1 — Quality Analysis: Single Agent Baseline

To understand the value of multi-agent conversation, let's compare with a single agent answering the same questions directly.

In [ ]:
# Single agent baseline comparison
single_agent_questions = [
    (
        "Researcher vs Skeptic topic",
        "Do large language models exhibit emergent abilities at certain scale thresholds? "
        "Analyze the evidence for and against this claim, considering potential confounders."
    ),
    (
        "Teacher vs Student topic",
        "Explain how neural networks learn through backpropagation in a clear, "
        "step-by-step way that a beginner could understand."
    ),
    (
        "Planner vs Critic topic",
        "Create a detailed plan for building a recommendation system for an e-commerce platform. "
        "Include phases, timelines, risks, and success metrics."
    ),
]

print("SINGLE AGENT BASELINE")
print("=" * 70)

single_results = []
for topic, question in single_agent_questions:
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Provide thorough, well-reasoned responses."
        )},
        {"role": "user", "content": question}
    ]

    start = time.time()
    response = generate(messages, max_new_tokens=500)
    elapsed = time.time() - start

    word_count = len(response.split())
    single_results.append({
        "topic": topic,
        "words": word_count,
        "time": round(elapsed, 1)
    })

    print(f"\n--- {topic} ---")
    print(f"Words: {word_count}, Time: {elapsed:.1f}s")
    print(f"Response: {response[:300]}...")

# Compare
print(f"\n\n{'='*70}")
print("MULTI-AGENT vs SINGLE AGENT COMPARISON")
print(f"{'='*70}")
print(f"{'Topic':<30s} {'Multi Words':>12s} {'Single Words':>13s} {'Multi Time':>11s} {'Single Time':>12s}")
print("-" * 80)
for (name, result), single in zip(experiments, single_results):
    print(f"{name:<30s} {result['total_words']:>12d} {single['words']:>13d} "
          f"{result['elapsed_seconds']:>10.1f}s {single['time']:>11.1f}s")

## 9. Shared vs Separate Context

An important design choice: should agents share a common conversation history, or should each agent only see messages addressed to them?

- **Shared context**: Both agents see the complete conversation. Simpler, but agents may echo each other.
- **Separate context**: Each agent only sees its own messages. More diverse perspectives, but may lose coherence.

Let's experiment.

In [ ]:
# Experiment: Separate context agents (each only sees their own messages)

class IsolatedAgent(ConversableAgent):
    """Agent that only sees messages it sent or received directly."""

    def _build_messages(self, conversation_with=None):
        """Build messages from only direct conversation."""
        messages = [{"role": "system", "content": self.system_prompt}]

        # Only include messages between this agent and the target
        for msg in self.history:
            if msg.sender == self.name:
                messages.append({"role": "assistant", "content": msg.content})
            elif msg.recipient == self.name:
                messages.append({"role": "user", "content": msg.content})
            # Skip messages not involving this agent

        return messages

# Run same Researcher vs Skeptic with isolated context
bus_isolated = MessageBus()

researcher_iso = IsolatedAgent(
    name="Researcher",
    system_prompt=(
        "You are a scientific researcher presenting findings. You make evidence-based claims, "
        "cite studies, and defend your position with logical arguments. "
        "Keep each response to 3-5 sentences."
    ),
    bus=bus_isolated,
    max_tokens=250
)

skeptic_iso = IsolatedAgent(
    name="Skeptic",
    system_prompt=(
        "You are a scientific skeptic. Challenge claims, ask for evidence, identify logical "
        "fallacies, and demand rigorous proof. "
        "Keep each response to 3-5 sentences."
    ),
    bus=bus_isolated,
    max_tokens=250
)

chat_iso = TwoAgentChat(researcher_iso, skeptic_iso, max_turns=6)
result_iso = chat_iso.run(
    "I believe that large language models exhibit emergent abilities at certain scale "
    "thresholds — capabilities that appear suddenly rather than gradually."
)

print(f"\n\nCOMPARISON: Shared vs Isolated Context")
print(f"  Shared context:   {result1['total_turns']} turns, {result1['total_words']} words")
print(f"  Isolated context: {result_iso['total_turns']} turns, {result_iso['total_words']} words")

## 10. Key Takeaways

### What We Built

| Component | Purpose |
|-----------|---------|
| **Message** | Structured inter-agent communication with metadata |
| **MessageBus** | Central routing with history and pub/sub |
| **ConversableAgent** | Agent with personality, history, and dialogue ability |
| **TwoAgentChat** | Turn-taking orchestration with termination detection |

### Key Insights

1. **Adversarial dialogue improves quality** — the Skeptic forced the Researcher to provide better evidence and acknowledge limitations
2. **Role-based conversation produces structure** — Teacher/Student dialogue naturally created clear, progressive explanations
3. **Iterative refinement works** — the Planner/Critic dialogue produced a more robust plan than a single pass
4. **Multi-agent is more expensive** — more turns = more LLM calls = more compute and time
5. **Shared context is simpler** — isolated context can diverge; shared context maintains coherence
6. **Termination is hard** — detecting when agents have reached a useful conclusion requires careful heuristics

### When to Use Multi-Agent Conversation

| Use Case | Recommended Pattern |
|----------|-------------------|
| Need rigorous analysis | Researcher + Skeptic |
| Need clear explanations | Teacher + Student |
| Need plan refinement | Planner + Critic |
| Need code review | Developer + Reviewer |
| Simple Q&A | Single agent (cheaper) |

### Design Principles

- **Keep system prompts focused** — each agent should have a clear, specific role
- **Limit turn count** — diminishing returns after 6-10 turns for most tasks
- **Monitor for loops** — agents can get stuck repeating similar points
- **Log everything** — the conversation trace is the debug tool

---

**Next Notebook**: [18_agent_debate_and_consensus.ipynb](./18_agent_debate_and_consensus.ipynb) — scaling from 2 agents to N agents in structured debates.